In [8]:

import pandas as pd
from agents import *
from papers import *
import time

start = time.time()
print(f'Start Time: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(start))}')

df = pd.read_csv('data/example/example_data.csv')
accessions = df['Accession']

for accession in accessions:

    print(f'Beginning Classification for {accession}')
    papers = os.listdir(f'data/library/{accession}')
    host_high_confidence = False  # Control flag

    with open(f'data/library/{accession}/{accession}.json', 'r') as f:
        metadata = json.load(f)

    for paper in papers:
        print(f'Beginning Classification for {accession}/{paper}')
        with open(f'data/library/{accession}/{paper}') as f:
            paper_text = f.read()

        state = {
            'paper_text': paper_text,
            'source_path': f'data/library/{accession}/{paper}',
            'model': 'qwen3.5',
            'metadata': metadata,
            'host_species': None,
            'reasoning': None,
            'confidence': None,
            'thermal_range': None,
            'temperature': None,
            'thermal_reasoning': None,
            'thermal_confidence': None,
        }

        host_classification = IdentifyHost(state)


        if host_classification.get('host_species') != 'unknown' and host_classification.get('confidence') == 'high':
            with open('data/example/host_classification.csv', 'a') as f:
                f.write(
                    f'{accession},{paper},{host_classification["host_species"]},{host_classification["reasoning"]},  {host_classification["confidence"]}\n')
            host_high_confidence = True
            state['host_species'] = host_classification['host_species']
            print(f'High-confidence classification for {accession}/{paper}\n{host_classification}')
            break  # 🚀 STOP processing papers for this accession

    if not host_high_confidence:
        with open('data/example/classification.txt', 'a') as f:
            f.write(f'Accession: {accession}\n')
            f.write('No high-confidence result found.\n\n')

end=time.time()


print(f'End Time: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(end))}')
print(f'Total Time: {end - start:.2f} seconds')

Start Time: 2026-04-24 13:59:43
Beginning Classification for ADI87650
Beginning Classification for ADI87650/41539756.txt
Beginning Classification for ADI87650/PMC12893549.1.txt
Beginning Classification for ADI87650/41932316.txt
Beginning Classification for ADI87650/PMC12930573.1.txt
High-confidence classification for ADI87650/PMC12930573.1.txt
{'host_species': 'Acinetobacter baumannii', 'reasoning': "The host species is explicitly stated in the paper text. The text identifies the host strain as 'Acinetobacter baumannii M13 isolate' and states the phage targets 'carbapenem-resistant Acinetobacter baumannii (CRAB)'. Specifically, the Methods section notes: 'Acinetobacter baumannii M13 isolate recovered from a wound infection... was selected as the host strain'. The Results section confirms: 'A novel lytic Acinetobacter phage vB_AbaM_MU1 was isolated... It showed lytic efficiency against 9/17 CRAB strains.'", 'confidence': 'high'}
Beginning Classification for WXH46553
Beginning Classifica

In [2]:
# imports


import os
import pandas as pd
from agents import *
import time

start = time.time()
print(f'Start Time: {time.strftime("%Y-%m-%d %H:%M:%S" ,time.localtime(start))}')

df = pd.read_csv('data/example/example_data.csv')
accessions = df['Accession']


for accession in accessions:
    print(f'Beginning Classification for {accession}')
    papers = os.listdir(f'data/library/{accession}')
    thermal_high_confidence = False  # Control flag

    # Begin writing all reasoning to file
    with open(f'data/example/classifications/{accession}_thermal_classification.txt', 'a') as f:
        f.write(f'###############{accession}###############\nStart time: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(time.time()))}\n')

    with open(f'data/library/{accession}/{accession}.json', 'r') as f:
        metadata = json.load(f)

    for paper in papers:
        print(f'Beginning Classification for {accession}/{paper}')
        with open(f'data/library/{accession}/{paper}') as f:
            paper_text = f.read()

        state = {
            'paper_text': paper_text,
            'source_path': f'data/library/{accession}/{paper}',
            'model': 'qwen3.5',
            'metadata': metadata,
            'host_species': df.loc[df['Accession'] == accession, 'Host'].iloc[0],
            'reasoning': None,
            'confidence': None,
            'thermal_range': None,
            'temperature': None,
            'thermal_reasoning': None,
            'thermal_confidence': None,
        }

        thermal_classification = IdentifyThermalRange(state)
        print(f"LLM Output: {thermal_classification}")
        with open(f'data/example/classifications/{accession}_thermal_classification.txt', 'a') as f:
            f.write(f'===============================================================\n'
                    f'Paper: {paper}\n'
                    f'Thermal Range: {thermal_classification.get("thermal_range")}\n'
                    f'Temperature: {thermal_classification.get("temperature")}\n'
                    f'Reasoning: {thermal_classification.get("thermal_reasoning")}\n'
                    f'Confidence: {thermal_classification.get("thermal_confidence")}\n'
                    f'Time: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(time.time()))}\n\n')

        if thermal_classification.get('thermal_range') != 'unknown' and thermal_classification.get('thermal_confidence') == 'high':
            with open('data/example/thermal_classification.csv', 'a') as f:
                f.write(
                    f'{accession},{paper},{thermal_classification.get("thermal_range")},{thermal_classification.get("temperature")},{thermal_classification.get("thermal_reasoning")}\n'
                )
            thermal_high_confidence = True
            print(f'High-confidence classification for {accession}/{paper}\n{thermal_classification}\n\n')
            break  # 🚀 STOP processing papers for this accession

    if not thermal_high_confidence:
        with open(f'data/example/thermal_classification.csv', 'a') as f:
            f.write(f'{accession},None,Unknown,Unknown,{thermal_classification.get("thermal_reasoning")}\n')
            f.write('No high-confidence result found.\n\n')

end=time.time()


print(f'End Time: {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(end))}')
print(f'Total Time: {end - start:.2f} seconds')

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Start Time: 2026-04-26 11:42:58
Beginning Classification for ADI87650
Beginning Classification for ADI87650/41539756.txt
LLM Output: {'thermal_range': 'unknown', 'temperature': 'unknown', 'thermal_reasoning': "The paper states: 'Bacteriophages are promising antibacterial agents across the farm-to-fork continuum... We evaluated chitosan concentration effects on grafting efficiency, phage viability in simulated gastrointestinal (GI) conditions...' However, no specific temperature values or thermal growth ranges are explicitly stated in the text.", 'thermal_confidence': 'high'}
Beginning Classification for ADI87650/PMC12893549.1.txt
LLM Output: {'thermal_range': 'unknown', 'temperature': 'unknown', 'thermal_reasoning': "The paper states 'Aliquots of nucleic acid were added to the protein in a thermostated (20oC) SPEX Fluoromax 2 spectrofluorimeter', indicating experimental conditions rather than organismal thermal characteristics, and no growth temperature or thermal classification is pro

In [12]:
df = pd.read_csv('data/example/example_data.csv')
accessions = df['Accession']

'QAY18187'